# 6th attempt - RCNN - 5

In [22]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [23]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [24]:
SIZE = 5
AMOUNT_BOARDS = 10000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2
gen_data = gen-1

In [25]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 29760
len train:  24105
len val:  2679
len test:  2976


In [26]:
X_train.shape

(24105, 25)

In [27]:
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print(X_train_array.shape)
print(y_train_array.shape)
print(X_val_array.shape)
print(y_val_array.shape)
print(X_test_array.shape)
print(y_test_array.shape)

(24105, 5, 5, 1)
(24105, 1)
(2679, 5, 5, 1)
(2679, 1)
(2976, 5, 5, 1)
(2976, 1)


In [28]:
model,history = build_RCNN_softmax(gen, X_train_array, y_train_array, SIZE, batch_size=32, epochs=3)

X_train shape: (24104, 1, 5, 5, 1)
y_train shape: (24104, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.6716 - loss: 0.6201 - val_accuracy: 0.6594 - val_loss: 0.6266
Epoch 2/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.6924 - loss: 0.5833 - val_accuracy: 0.6884 - val_loss: 0.5853
Epoch 3/3
603/603 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.7097 - loss: 0.5667 - val_accuracy: 0.6897 - val_loss: 0.5865


In [29]:
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test,), dtype='float32')
for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data].reshape(gen_data, SIZE, SIZE, 1)
    y_test[i] = y_test_array[i]

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

 8/93 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6576 - loss: 0.5891 

C:\Users\דרור\AppData\Local\Temp\ipykernel_1768\2954292026.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_test[i] = y_test_array[i]


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6682 - loss: 0.6058
Test accuracy: 0.677


In [30]:
import matplotlib.pyplot as plt
import numpy as np

def plot_results(y_test, y_pred):
    """
    מצייר פיזור בין הפלט האמיתי לפלט החזוי.
    """
    y_test = y_test.flatten()
    y_pred = y_pred.flatten()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    
    # קו אידיאלי (תחזית = אמת)
    plt.plot([0, 1], [0, 1], linestyle='--')

    plt.xlabel("True Value (y_test)")
    plt.ylabel("Predicted Value (y_pred)")
    plt.title("Prediction Scatter Plot")
    plt.grid(True)
    plt.show()


In [31]:
# יצירת סט בדיקה
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test, 1), dtype='float32')

for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data].reshape(gen_data, SIZE, SIZE, 1)
    y_test[i] = y_test_array[i]

# הפעלת פונקציית ההערכה
y_test_flat = y_test.flatten()
evaluate_model(model, X_test, y_test_flat)


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        334 │        687 │
│ True=Dead    │        273 │       1681 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.677
Precision   : 0.550
Recall      : 0.327
F1-score    : 0.410


In [33]:
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
for i in range(SIZE*SIZE): # to any pixel in the expected board
    name_col = 'Col_' + str(i+amount_features)
    target = reverse_df_sort[name_col]
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
    
    X_train_array = X_train.to_numpy()
    y_train_array = y_train.to_numpy()
    X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
    y_train_array = y_train_array.reshape((y_train.shape[0],1))
    
    model = build_RCNN_softmax(gen, X_train_array, y_train_array, SIZE, 32, 3)
    name_file = f"{PATH_MODELS}\\model6_RCNN_softmax\{SIZE}\\size_{SIZE}_pixel_{str(i+1)}.pkl"
    with open(name_file, 'wb') as file:
        pickle.dump(model, file)
    print(i)

X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
Epoch 1/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.6733 - loss: 0.6199 - val_accuracy: 0.6728 - val_loss: 0.6168
Epoch 2/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.7044 - loss: 0.5759 - val_accuracy: 0.6912 - val_loss: 0.5833
Epoch 3/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.7162 - loss: 0.5582 - val_accuracy: 0.6963 - val_loss: 0.5799
0
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
Epoch 1/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.6696 - loss: 0.6200 - val_accuracy: 0.6657 - val_loss: 0.6189
Epoch 2/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.6974 - loss: 0.5831 - val_accuracy: 0.6952 - val_loss: 0.5931
Epoch 3/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 39s 31ms/step - accuracy: 0.7122 - loss: 0.5638 - val_accuracy: 0.6924 - val_loss: 0.5847
1
X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)
Epoch 1/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 